In [2]:
import os
import re
import glob
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt

In [3]:
def plot_lv_rv_scatter_with_overlay(
    df_bg,
    df_new=None,
    lv_col="侧脑室（左）",
    rv_col="侧脑室（右）",
    unit="mm³",
    scale=1.0,                 # 若 df 是 ml 想画 mm^3：scale=1000.0
    bg_label="Reference Data",
    new_label="NJGL Samples ",
    bg_kwargs=None,
    new_kwargs=None,
    save_path=None,
):
    """
    背景散点(df_bg) + 叠加新样本(df_new)（不同颜色/样式）
    - df_new 可为 None；也可传 DataFrame
    - 如果你新样本是“点列/列表”，用下面的辅助方式：df_new = pd.DataFrame({lv_col: x_new, rv_col: y_new})
    """
    if bg_kwargs is None:
        bg_kwargs = dict(s=60, alpha=0.35, label=bg_label)
    if new_kwargs is None:
        new_kwargs = dict(s=90, alpha=0.95, marker="*", edgecolors="k", linewidths=0.6, label=new_label)

    def _xy_from_df(d):
        x = d[lv_col].to_numpy(dtype=float) * float(scale)
        y = d[rv_col].to_numpy(dtype=float) * float(scale)
        m = np.isfinite(x) & np.isfinite(y)
        return x[m], y[m]

    x_bg, y_bg = _xy_from_df(df_bg)

    plt.figure(figsize=(7, 6))
    plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False

    # 背景
    plt.scatter(x_bg, y_bg, **bg_kwargs)

    # 叠加新样本
    if df_new is not None:
        x_new, y_new = _xy_from_df(df_new)
        plt.scatter(x_new, y_new, **new_kwargs)

    plt.xlabel(f"Left Ventricle Volume ({unit})")
    plt.ylabel(f"Right Ventricle Volume ({unit})")
    plt.title("LV vs RV Volume Scatter")
    plt.grid(alpha=0.3)
    plt.legend()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved: {save_path}")
    else:
        plt.show()

In [4]:
def plot_lv_rv_scatter_with_overlay_and_labels(
    df_bg,
    df_new,
    name_map,                         # dict: {subj_names: "中文名"}
    lv_col="侧脑室（左）",
    rv_col="侧脑室（右）",
    subj_col="subj_names",
    unit="mm³",
    scale=1.0,                        # ml->mm^3: 1000.0
    bg_kwargs=None,
    new_kwargs=None,
    text_kwargs=None,
    save_path=None,
):
    """
    背景 df_bg + 叠加 df_new，并给 df_new 每个点标注中文名称（由 name_map 提供）
    """
    if bg_kwargs is None:
        bg_kwargs = dict(s=60, alpha=0.25, label="Reference Data")
    if new_kwargs is None:
        new_kwargs = dict(s=90, alpha=0.95, marker="*", edgecolors="k", linewidths=0.6, label="NJGL Samples")
    if text_kwargs is None:
        text_kwargs = dict(fontsize=9)  # 可自行加 bbox 等

    def _xy(d):
        x = d[lv_col].to_numpy(dtype=float) * float(scale)
        y = d[rv_col].to_numpy(dtype=float) * float(scale)
        m = np.isfinite(x) & np.isfinite(y)
        return x[m], y[m], m

    # 背景点
    x_bg, y_bg, _ = _xy(df_bg)

    plt.figure(figsize=(7, 6))
    plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 若中文显示乱码，改为你系统可用中文字体，如 SimHei
    plt.rcParams["axes.unicode_minus"] = False

    plt.scatter(x_bg, y_bg, **bg_kwargs)

    # 新点 + 标注
    x_new, y_new, m_new = _xy(df_new)
    df_new_valid = df_new.loc[m_new].copy()

    plt.scatter(x_new, y_new, **new_kwargs)

    # 给每个点标注
    for xi, yi, subj in zip(x_new, y_new, df_new_valid[subj_col].astype(str).tolist()):
        label = name_map.get(subj, subj)  # 找不到就用 subj_names 兜底
        plt.annotate(
            label,
            (xi, yi),
            xytext=(5, 5),
            textcoords="offset points",
            **text_kwargs
        )

    plt.xlabel(f"Left Ventricle Volume ({unit})")
    plt.ylabel(f"Right Ventricle Volume ({unit})")
    plt.title("LV vs RV Volume Scatter")
    plt.grid(alpha=0.3)
    plt.legend()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved: {save_path}")
    else:
        plt.show()


# ===== 用法示例 =====
# name_map = {
#   "20240605153643-mask": "病例A（中文）",
#   "20231221105024-mask": "病例B（中文）",
# }
# plot_lv_rv_scatter_with_overlay_and_labels(
#     df_bg=df,
#     df_new=df_new,
#     name_map=name_map,
#     scale=1.0,
#     save_path="/mnt/sda/public/.../lv_rv_overlay_labeled.png"
# )


In [4]:
gfy_data_df = pd.read_csv("/mnt/sda/public/00_Projects/2512_AutoRecon/results/gfy_results/gfy_surface_vol_all.csv")
njgl_data_df = pd.read_csv("/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/volume_stats.csv")
plot_lv_rv_scatter_with_overlay(df_bg=gfy_data_df,df_new=None, scale=1000.0,
                                save_path="/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/LV_vs_RV_scatter_gfy_default.png"
)
plot_lv_rv_scatter_with_overlay(
    df_bg=gfy_data_df, df_new=njgl_data_df, scale=1000.0, 
    save_path="/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/LV_vs_RV_scatter_overlay_raw.png"
)


Saved: /mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/LV_vs_RV_scatter_gfy_default.png
Saved: /mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/LV_vs_RV_scatter_overlay_raw.png


In [5]:
gfy_data_df_v2 = pd.read_csv("/mnt/sda/public/00_Projects/2512_AutoRecon/results/gfy_results/surface_vol_stats_drop_diff_outliers_final.csv")
plot_lv_rv_scatter_with_overlay(df_bg=gfy_data_df_v2,df_new=None, scale=1000.0,
                                save_path="/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/LV_vs_RV_scatter_gfy_default_final.png"
)
plot_lv_rv_scatter_with_overlay(
    df_bg=gfy_data_df_v2, df_new=njgl_data_df, scale=1000.0, 
    save_path="/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/LV_vs_RV_scatter_overlay_final.png"
)


Saved: /mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/LV_vs_RV_scatter_gfy_default_final.png
Saved: /mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/LV_vs_RV_scatter_overlay_final.png


In [6]:
name_map = {
    "01":"蔡琦琦",
    "02":"洪文文",
    "03":"蒋唯",
    "06":"李欣函",
    "07":"沈雪",
    "08":"夏伟瑜",
    "09":"杨金枝",
    "10":"俞前",
    "11_zwy":"赵皖豫",
    "12":"赵尹影",
    "13":"周梦汝"
}
name_map = {
    "01":"cqq",
    "02":"hww",
    "03":"jw",
    "06":"lxh",
    "07":"shx",
    "08":"xwy",
    "09":"yjz",
    "10":"yq",
    "11_zwy":"zwy",
    "12":"zhyy",
    "13":"zhmr"
}
plot_lv_rv_scatter_with_overlay_and_labels(
    df_bg=gfy_data_df_v2,
    df_new=njgl_data_df,
    name_map=name_map,
    scale=1000.0,
    save_path="/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/lv_rv_overlay_labeled_final.png"
)

Saved: /mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/lv_rv_overlay_labeled_final.png


In [ ]:
gfy_data_df_v2 = pd.read_csv("/mnt/sda/public/00_Projects/2512_AutoRecon/results/gfy_results/surface_vol_stats_healthy.csv")
njgl_data_df = pd.read_csv("/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/volume_stats.csv")
name_map = {
    "01":"cqq",
    "02":"hww",
    "03":"jw",
    "06":"lxh",
    "07":"shx",
    "08":"xwy",
    "09":"yjz",
    "10":"yq",
    "11_zwy":"zwy",
    "12":"zhyy",
    "13":"zhmr"
}
plot_lv_rv_scatter_with_overlay(
    df_bg=gfy_data_df_v2, df_new=njgl_data_df, scale=1000.0, 
    save_path="/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/lv_rv_overlay_final_v2.png"
)
plot_lv_rv_scatter_with_overlay_and_labels(
# plot_lv_rv_scatter_with_overlay(
    df_bg=gfy_data_df_v2,
    df_new=njgl_data_df,
    name_map=name_map,
    scale=1000.0,
    save_path="/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/lv_rv_overlay_labeled_final_v2.png"
)

Saved: /mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/lv_rv_overlay_labeled_final_v2.png


In [8]:
def plot_4scatter_with_fit(
    df,
    x_cols, y_cols,
    titles,
    xlabels=None, ylabels=None,
    colors=None, fit_colors=None,
    suptitle=None,
    save_path=None,
    dpi=300,
    y_scale=1.0,   # <<< 新增：把 y 乘以这个比例（ml->mm^3 就是 1000）
):
    assert len(x_cols) == len(y_cols) == len(titles) == 4

    if xlabels is None: xlabels = x_cols
    if ylabels is None: ylabels = y_cols
    if colors is None: colors = ["skyblue", "lightcoral", "mediumseagreen", "orange"]
    if fit_colors is None: fit_colors = ["r", "b", "r", "b"]

    plt.figure(figsize=(12, 10))
    plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False

    zs = []
    for i in range(4):
        ax = plt.subplot(2, 2, i + 1)

        x = df[x_cols[i]].to_numpy(dtype=float)
        y = df[y_cols[i]].to_numpy(dtype=float) * float(y_scale)  # <<< 统一缩放

        m = np.isfinite(x) & np.isfinite(y)
        x, y = x[m], y[m]

        ax.scatter(x, y, color=colors[i], s=60, alpha=0.8)

        if len(x) >= 2:
            z = np.polyfit(x, y, 1)
            p = np.poly1d(z)
            xs = np.linspace(x.min(), x.max(), 200)
            ax.plot(xs, p(xs), "--", color=fit_colors[i], alpha=0.6,
                    label=f'Fit: y={z[0]:.5f}x + {z[1]:.2f}')
            zs.append(z)
        else:
            zs.append([np.nan, np.nan])

        ax.set_title(titles[i], fontsize=12)
        ax.set_xlabel(xlabels[i], fontsize=10)
        ax.set_ylabel(ylabels[i], fontsize=10)
        ax.grid(alpha=0.3)
        ax.legend()

    if suptitle:
        plt.suptitle(suptitle, fontsize=16, y=0.98)
        plt.tight_layout(rect=[0, 0, 1, 0.95])
    else:
        plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
        plt.close()
        print(f"图片已保存至：{save_path}")
    else:
        plt.show()

    print("\n各结构的拟合曲线表达式：")
    for i, z in enumerate(zs):
        print(f"{titles[i]}：y = {z[0]:.5f} * {x_cols[i]} + {z[1]:.2f}")

    return zs



In [ ]:

x_cols = ["wm_area_l_list", "wm_area_r_list", "cortex_area_l_list", "cortex_area_r_list"]
y_cols = ["侧脑室（左）", "侧脑室（右）", "侧脑室（左）", "侧脑室（右）"]

titles = [
    "LV (mm³) vs Left WM Area",
    "RV (mm³) vs Right WM Area",
    "LV (mm³) vs Left Cortex Area",
    "RV (mm³) vs Right Cortex Area",
]

plot_4scatter_with_fit(
    gfy_data_df_v2,
    x_cols=x_cols,
    y_cols=y_cols,
    titles=titles,
    xlabels=["Surface Area (mm²)"] * 4,
    ylabels=["Volume (mm³)"] * 4,
    suptitle="Ventricles Volume (mm³) vs Surface Area (mm²)",
    save_path="/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/ventricle_vol_vs_area_gfy_default.png",
    y_scale=1000.0,   # <<< 关键：ml -> mm^3
)

图片已保存至：/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/ventricle_vol_vs_area_gfy_default.png

各结构的拟合曲线表达式：
LV (mm³) vs Left WM Area：y = 0.05979 * wm_area_l_list + 1953.22
RV (mm³) vs Right WM Area：y = 0.09437 * wm_area_r_list + 1381.24
LV (mm³) vs Left Cortex Area：y = 0.07670 * cortex_area_l_list + 1739.10
RV (mm³) vs Right Cortex Area：y = 0.11443 * cortex_area_r_list + 1115.29


[array([5.97855771e-02, 1.95322283e+03]),
 array([9.43697016e-02, 1.38123784e+03]),
 array([7.66961802e-02, 1.73910352e+03]),
 array([1.14434956e-01, 1.11528617e+03])]

In [29]:
def plot_4scatter_with_fit(
    df,
    x_cols, y_cols,
    titles,
    xlabels=None, ylabels=None,
    colors=None, fit_colors=None,
    suptitle=None,
    save_path=None,
    dpi=300,
    y_scale=1.0,          # y 的缩放（ml->mm^3: 1000）
    new_point=None,       # <<< 新增：dict，结构见你给的 new_point
    new_point_label=None,
    new_point_kwargs=None # <<< 新增：新点样式
):
    assert len(x_cols) == len(y_cols) == len(titles) == 4

    if xlabels is None: xlabels = x_cols
    if ylabels is None: ylabels = y_cols
    if colors is None: colors = ["skyblue", "lightcoral", "mediumseagreen", "orange"]
    if fit_colors is None: fit_colors = ["r", "b", "r", "b"]
    if new_point_kwargs is None:
        new_point_kwargs = dict(s=140, marker="*", color="black", edgecolors="white", linewidths=0.8, zorder=10)

    plt.figure(figsize=(12, 10))
    plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False

    zs = []
    for i in range(4):
        ax = plt.subplot(2, 2, i + 1)

        x = df[x_cols[i]].to_numpy(dtype=float)
        y = df[y_cols[i]].to_numpy(dtype=float) * float(y_scale)

        m = np.isfinite(x) & np.isfinite(y)
        x, y = x[m], y[m]

        ax.scatter(x, y, color=colors[i], s=60, alpha=0.8, label=f'Reference Data')

        # 新点：按当前子图的 x/y 字段取值
        if new_point is not None:
            x_new = float(new_point[x_cols[i]])
            y_new = float(new_point[y_cols[i]]) * float(y_scale)
            if np.isfinite(x_new) and np.isfinite(y_new):
                # ax.scatter([x_new], [y_new], **new_point_kwargs)
                ax.scatter(
                [x_new], [y_new],
                color='red', s=200, marker='*', alpha=1.0, label=f'Subject-{new_point_label}')
                # ax.annotate(
                #     new_point_label,
                #     (x_new, y_new),
                #     xytext=(6, 6),
                #     textcoords="offset points",
                #     fontsize=9,
                #     color=new_point_kwargs.get("color", "black"),
                # )
                
        # 拟合线
        if len(x) >= 2:
            z = np.polyfit(x, y, 1)
            p = np.poly1d(z)
            xs = np.linspace(x.min(), x.max(), 200)
            ax.plot(xs, p(xs), "--", color=fit_colors[i], alpha=0.6,
                    label=f'Fit: y={z[0]:.5f}x + {z[1]:.2f}')
            zs.append(z)
        else:
            zs.append([np.nan, np.nan])
        ax.set_title(titles[i], fontsize=12)
        ax.set_xlabel(xlabels[i], fontsize=10)
        ax.set_ylabel(ylabels[i], fontsize=10)
        ax.grid(alpha=0.3)
        ax.legend()
        


    if suptitle:
        plt.suptitle(suptitle, fontsize=16, y=0.98)
        plt.tight_layout(rect=[0, 0, 1, 0.95])
    else:
        plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
        plt.close()
        print(f"图片已保存至：{save_path}")
    else:
        plt.show()

    print("\n各结构的拟合曲线表达式：")
    for i, z in enumerate(zs):
        print(f"{titles[i]}：y = {z[0]:.5f} * {x_cols[i]} + {z[1]:.2f}")

    return zs


# ===================== 你给的 new_point 结构示例 =====================
# new_point = {
#     "wm_area_l": 12345,
#     "wm_area_r": 12001,
#     "wm_vol_ml_l": 2.3,
#     "wm_vol_ml_r": 2.1,
#     "cortex_area_l": 54321,
#     "cortex_area_r": 52000,
#     "mass_vol_ml_l": 12.3,
#     "mass_vol_ml_r": 11.8,
#     "侧脑室（左）": 3.1,
#     "侧脑室（右）": 2.9,
# }

# 例：如果你画的是 “Area vs Volume(ml)” 并希望显示 mm^3，则 y_scale=1000
# x_cols/y_cols 必须是 new_point 里存在的键名（或 df 列名一致）
# x_cols = ["wm_area_l", "wm_area_r", "cortex_area_l", "cortex_area_r"]
# y_cols = ["侧脑室（左）", "侧脑室（右）", "侧脑室（左）", "侧脑室（右）"]
# plot_4scatter_with_fit(df, x_cols, y_cols, titles, y_scale=1000.0, new_point=new_point, new_point_label="Case-1")


In [13]:
from surface_stats import get_surface_area
def load_new_point(surface_file_dir):
    wm_area_l = get_surface_area(surface_file_dir,"lh.smoothwm.nofix")
    wm_area_r = get_surface_area(surface_file_dir,"rh.smoothwm.nofix")

    cortex_area_l = get_surface_area(surface_file_dir,"lh.smoothcortex.nofix")
    cortex_area_r = get_surface_area(surface_file_dir,"rh.smoothcortex.nofix")
    new_point = {
        "wm_area_l_list":wm_area_l,
        "wm_area_r_list":wm_area_r,
        "cortex_area_l_list":cortex_area_l,
        "cortex_area_r_list":cortex_area_r,
    }
    return new_point

In [32]:
plot_4scatter_with_fit(gfy_data_df_v2, 
                        x_cols, y_cols, 
                        titles, 
                        y_scale=1000.0, 
                        new_point=None, 
                        new_point_label=None,
                        xlabels=["Surface Area (mm²)"] * 4, 
                        ylabels=["Volume (mm³)"] * 4,
                        save_path=f"/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/LR_defualt.png")

图片已保存至：/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/LR_defualt.png

各结构的拟合曲线表达式：
LV (mm³) vs Left WM Area：y = 0.05979 * wm_area_l_list + 1953.22
RV (mm³) vs Right WM Area：y = 0.09437 * wm_area_r_list + 1381.24
LV (mm³) vs Left Cortex Area：y = 0.07670 * cortex_area_l_list + 1739.10
RV (mm³) vs Right Cortex Area：y = 0.11443 * cortex_area_r_list + 1115.29


[array([5.97855771e-02, 1.95322283e+03]),
 array([9.43697016e-02, 1.38123784e+03]),
 array([7.66961802e-02, 1.73910352e+03]),
 array([1.14434956e-01, 1.11528617e+03])]

In [ ]:
surface_dir = "/mnt/sda/public/00_Projects/2512_AutoRecon/data/surface"
sub_dir = ["00_zwy", "01_cqq", "02", "03", "06", "07", "08", "09", "10", "12", "13"]
id_list = ["11_zwy", "01", "02", "03", "06", "07", "08", "09", "10", "12", "13"]
# name_map = {
#     "01":"cqq",
#     "02":"hww",
#     "03":"jw",
#     "06":"lxh",
#     "07":"shx",
#     "08":"xwy",
#     "09":"yjz",
#     "10":"yq",
#     "11_zwy":"zwy",
#     "12":"zhyy",
#     "13":"zhmr"
# }
# id_list = ["01", "02", "03", "06", "07", "08", "09", "10", "11_zwy", "12", "13"]


x_cols = ["wm_area_l_list", "wm_area_r_list", "cortex_area_l_list", "cortex_area_r_list"]
y_cols = ["侧脑室（左）", "侧脑室（右）", "侧脑室（左）", "侧脑室（右）"]
cnt = 0
for subj in sub_dir:
    new_point = load_new_point(f"{surface_dir}/{sub_dir[cnt]}")
    rows = njgl_data_df.loc[njgl_data_df["subj_names"]==id_list[cnt]]
    new_point["侧脑室（左）"] = rows["侧脑室（左）"]
    new_point["侧脑室（右）"] = rows["侧脑室（右）"]

    plot_4scatter_with_fit(gfy_data_df_v2, 
                           x_cols, y_cols, 
                           titles, 
                           y_scale=1000.0, 
                           new_point=new_point, 
                           new_point_label=name_map[id_list[cnt]],
                           xlabels=["Surface Area (mm²)"] * 4, 
                           ylabels=["Volume (mm³)"] * 4,
                           save_path=f"/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/Subj_{subj}_LR.png")
    cnt += 1
    # break

/tmp/ipykernel_1610678/1555711163.py:43: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  y_new = float(new_point[y_cols[i]]) * float(y_scale)


图片已保存至：/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/Subj_00_zwy_LR.png

各结构的拟合曲线表达式：
LV (mm³) vs Left WM Area：y = 0.05979 * wm_area_l_list + 1953.22
RV (mm³) vs Right WM Area：y = 0.09437 * wm_area_r_list + 1381.24
LV (mm³) vs Left Cortex Area：y = 0.07670 * cortex_area_l_list + 1739.10
RV (mm³) vs Right Cortex Area：y = 0.11443 * cortex_area_r_list + 1115.29
图片已保存至：/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/Subj_01_cqq_LR.png

各结构的拟合曲线表达式：
LV (mm³) vs Left WM Area：y = 0.05979 * wm_area_l_list + 1953.22
RV (mm³) vs Right WM Area：y = 0.09437 * wm_area_r_list + 1381.24
LV (mm³) vs Left Cortex Area：y = 0.07670 * cortex_area_l_list + 1739.10
RV (mm³) vs Right Cortex Area：y = 0.11443 * cortex_area_r_list + 1115.29
图片已保存至：/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/Subj_02_LR.png

各结构的拟合曲线表达式：
LV (mm³) vs Left WM Area：y = 0.05979 * wm_area_l_list + 1953.22
RV (mm³) vs Right WM Area：y = 0.09437 * wm_area_r_list + 1381.24
LV (mm³) vs 